# Sprint 2: Exploratory Data Analysis & Predictive Dataset Validation

This notebook analyzes the engineered feature store combining:
- Market price bars and technical indicators (`ta` library: RSI, MACD, EMA20/50, Bollinger Bands, ATR)
- SPY market regime benchmark (`spy_return_5d`)
- Daily FinBERT sentiment aggregates (`avg_sentiment`, `sentiment_ema_3`, `sentiment_delta`)
- Supervised 5-day directional target (`target = future_return_5d > 0`)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

csv_path = Path("../data/training_dataset.csv")
df = pd.read_csv(csv_path)
print(f"Loaded training dataset with shape: {df.shape}")
df.head()

## 1. Data Integrity & Missing Values

In [ ]:
print("Duplicate (ticker, date) rows:", df.duplicated(subset=["ticker", "date"]).any())
print("\nMissing values count:")
print(df.isnull().sum())

## 2. Target Class Balance (BUY vs NOT BUY)

In [ ]:
counts = df["target"].value_counts()
percentages = df["target"].value_counts(normalize=True) * 100
balance_df = pd.DataFrame({"Count": counts, "Percentage (%)": percentages.round(2)})
balance_df.index = ["Class 1 (BUY)", "Class 0 (NOT BUY)"]
balance_df

## 3. Feature Correlations with 5-Day Forward Return

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corrs = df[numeric_cols].corr()["future_return_5d"].sort_values(ascending=False)
print("Top Positive Correlations:")
print(corrs.head(6))
print("\nTop Negative Correlations:")
print(corrs.tail(6))

## 4. Baseline Logistic Regression Benchmark

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

features = ["avg_sentiment", "total_article_count", "rsi", "macd", "ema20", "volume_ratio"]
clean = df.dropna(subset=features + ["target"]).sort_values(by="date").reset_index(drop=True)

split_idx = int(len(clean) * 0.80)
train_df = clean.iloc[:split_idx]
test_df = clean.iloc[split_idx:]

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[features])
X_test = scaler.transform(test_df[features])
y_train = train_df["target"]
y_test = test_df["target"]

clf = LogisticRegression(class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Out-of-sample Directional Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["NOT BUY", "BUY"]))